# Warehouse Shipment Optimization using Linear Programming (PuLP)

## Business Problem
A company has multiple warehouses supplying multiple cities. Each warehouse has limited supply and each city has a fixed demand. Shipping cost differs for each route.

### Objective
**Minimize total transportation cost** while satisfying all city demands and not exceeding warehouse supply.

### Deliverables
- Problem setup
- LP model formulation
- Solution using PuLP
- Results table
- Insights and recommendations


## 1) Install & Import Libraries

In [ ]:
!pip install pulp

import pulp
import pandas as pd
import numpy as np

## 2) Define Data (Warehouses, Cities, Supply, Demand, Costs)

In [ ]:
warehouses = ["W1", "W2", "W3"]
cities = ["C1", "C2", "C3", "C4"]

# Supply capacity at each warehouse
supply = {
    "W1": 100,
    "W2": 120,
    "W3": 80
}

# Demand requirement at each city
demand = {
    "C1": 70,
    "C2": 90,
    "C3": 60,
    "C4": 80
}

# Transportation cost per unit for each (warehouse, city) route
cost = {
    ("W1", "C1"): 4, ("W1", "C2"): 6, ("W1", "C3"): 8, ("W1", "C4"): 13,
    ("W2", "C1"): 5, ("W2", "C2"): 4, ("W2", "C3"): 3, ("W2", "C4"): 9,
    ("W3", "C1"): 6, ("W3", "C2"): 7, ("W3", "C3"): 5, ("W3", "C4"): 6
}

print("Total Supply:", sum(supply.values()))
print("Total Demand:", sum(demand.values()))

## 3) Create the Linear Programming Model

We define:
- Decision variables: Units shipped from each warehouse to each city
- Objective: Minimize total cost
- Constraints:
  - Warehouse supply limits
  - City demand satisfaction


In [ ]:
# Create the LP problem (Minimization)
model = pulp.LpProblem("Warehouse_Shipment_Optimization", pulp.LpMinimize)

# Decision variables: Ship[w,c] = units shipped from warehouse w to city c
Ship = pulp.LpVariable.dicts(
    "Ship",
    ((w, c) for w in warehouses for c in cities),
    lowBound=0,
    cat="Continuous"
)

# Objective: Minimize total transportation cost
model += pulp.lpSum(cost[(w, c)] * Ship[(w, c)] for w in warehouses for c in cities)

# Supply constraints: total outgoing from warehouse <= supply
for w in warehouses:
    model += pulp.lpSum(Ship[(w, c)] for c in cities) <= supply[w], f"Supply_{w}"

# Demand constraints: total incoming to city == demand
for c in cities:
    model += pulp.lpSum(Ship[(w, c)] for w in warehouses) == demand[c], f"Demand_{c}"

print(model)

## 4) Solve the Optimization Model

In [ ]:
model.solve()

print("Status:", pulp.LpStatus[model.status])
print("Minimum Transportation Cost =", pulp.value(model.objective))

## 5) Display Optimal Shipment Plan

In [ ]:
results = []

for w in warehouses:
    for c in cities:
        shipped = Ship[(w, c)].value()
        if shipped is None:
            shipped = 0
        if shipped > 0:
            results.append([w, c, shipped, cost[(w, c)], shipped * cost[(w, c)]])

df_results = pd.DataFrame(results, columns=["Warehouse", "City", "Units Shipped", "Cost/Unit", "Total Cost"])
df_results

## 6) Warehouse Utilization Analysis
This shows how much each warehouse is being used compared to its supply capacity.

In [ ]:
utilization = []

for w in warehouses:
    total_out = sum(Ship[(w, c)].value() for c in cities)
    utilization.append([w, total_out, supply[w], (total_out / supply[w]) * 100])

df_util = pd.DataFrame(utilization, columns=["Warehouse", "Total Shipped", "Supply", "Utilization %"])
df_util

## 7) City Demand Fulfillment Check
This confirms each city received exactly the required demand.

In [ ]:
fulfillment = []

for c in cities:
    total_in = sum(Ship[(w, c)].value() for w in warehouses)
    fulfillment.append([c, total_in, demand[c]])

df_fulfill = pd.DataFrame(fulfillment, columns=["City", "Received", "Demand"])
df_fulfill

## 8) Key Insights & Recommendations

### Insights
- The model chooses the lowest-cost shipping routes to satisfy all city demands.
- Warehouses that have cheaper routes to high-demand cities tend to be used more.
- If a warehouse is fully utilized, it indicates a potential capacity bottleneck.

### Recommendations
- Increase supply capacity in the warehouse with the highest utilization.
- Negotiate transport contracts on routes with high cost.
- If demand increases, expand the warehouse that serves the most cities at low cost.
